# VGGT view-count reconstruction ablation

SPAR manifests (`dataset/convert_spar_sftqa.py:sample_frames`) cycle-pad scenes with fewer than `num_frames` images, and uniform-sample long scenes without regard to real pose diversity. This notebook tests the direct effect on VGGT geometry: for one or more scenes, sample 1..8 frames (same uniform rule as `sample_frames`) and run frozen VGGT, then compare depth confidence + camera-pose diversity as a function of view count.

Edit the `SCENE_DIRS` / `FRAME_COUNTS` cell below, then run top to bottom.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VGGT_ROOT = ROOT / "vggt"
for p in (ROOT, VGGT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

print("root:", ROOT)

root: /glob/g01-cache/pf/Yushuo/vjepa201


In [3]:
# ── config — edit this ──────────────────────────────────────────────────────
SCENE_DIRS = [
    ROOT / "source_data/spar/scannet/images/scene0002_00/image_color",
    # add more scene dirs to compare across scenes in one run
]
FRAME_COUNTS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 16, 24, 32]
VGGT_CKPT = ROOT / "ckpts/vggt.pt"
CONF_THRES = 5.0          # world_points_conf threshold for a "valid" 3D point
OUTPUT_DIR = ROOT / "outputs/vggt_view_ablation"
SAVE_POINTCLOUD = True
IMG_EXTS = (".jpg", ".jpeg", ".png")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("device:", device, "dtype:", dtype)

device: cpu dtype: torch.float16


## Helpers

`sample_frames` mirrors `dataset/convert_spar_sftqa.py:sample_frames` (uniform-index rule), minus the cycle-pad branch — this notebook never asks for more frames than a scene has.

In [7]:
def sample_frames(images: list[str], num_frames: int) -> list[str]:
    if num_frames == 1:
        return [images[len(images) // 2]]
    idxs = [round(i * (len(images) - 1) / (num_frames - 1)) for i in range(num_frames)]
    return [images[i] for i in idxs]


def camera_centers(extrinsic: np.ndarray) -> np.ndarray:
    """extrinsic: (S, 3, 4) world-to-cam [R|t] -> (S, 3) camera centers in world coords."""
    R = extrinsic[:, :3, :3]
    t = extrinsic[:, :3, 3]
    return -np.einsum("sij,si->sj", R, t)  # C = -R^T t, per-frame


def pairwise_pose_diversity(extrinsic: np.ndarray) -> tuple[float, float]:
    """Mean pairwise camera-center distance and mean pairwise rotation angle (deg)
    across all i<j view pairs. Returns (nan, nan) for a single view."""
    S = extrinsic.shape[0]
    if S < 2:
        return float("nan"), float("nan")
    R = extrinsic[:, :3, :3]
    centers = camera_centers(extrinsic)
    trans_dists, rot_degs = [], []
    for i in range(S):
        for j in range(i + 1, S):
            trans_dists.append(np.linalg.norm(centers[i] - centers[j]))
            R_ij = R[i].T @ R[j]
            cos_angle = np.clip((np.trace(R_ij) - 1.0) / 2.0, -1.0, 1.0)
            rot_degs.append(np.degrees(np.arccos(cos_angle)))
    return float(np.mean(trans_dists)), float(np.mean(rot_degs))


@torch.no_grad()
def run_vggt(model: VGGT, image_paths: list[str]) -> dict:
  images = load_and_preprocess_images(image_paths).to(device)  # (S, 3, H, W)
  with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda"):
      predictions = model(images)
  extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])
  predictions["extrinsic"] = extrinsic
  predictions["intrinsic"] = intrinsic
  # 不要再手动覆盖 predictions["images"] — model.forward() 内部已经存了
  # batched (1,S,3,H,W) 版本, squeeze(0) 才能对上
  return {k: (v.squeeze(0).float().cpu().numpy() if isinstance(v, torch.Tensor) else v) for k, v in predictions.items()}

def save_pointcloud(preds: dict, conf_thres: float, out_path: Path) -> int:
    import trimesh

    pts = preds["world_points"]          # (S, H, W, 3)
    conf = preds["world_points_conf"]    # (S, H, W)
    imgs = preds["images"]               # (S, 3, H, W) in [0, 1]

    mask = conf > conf_thres
    verts = pts[mask]
    if verts.shape[0] == 0:
        return 0
    colors = (imgs.transpose(0, 2, 3, 1)[mask] * 255).astype(np.uint8)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    trimesh.PointCloud(verts, colors=colors).export(str(out_path))
    return int(verts.shape[0])

## Load VGGT (once)

In [8]:
print(f"loading VGGT from {VGGT_CKPT} ...")
model = VGGT()
state = torch.load(VGGT_CKPT, map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.eval().to(device)
print("loaded.")

loading VGGT from /glob/g01-cache/pf/Yushuo/vjepa201/ckpts/vggt.pt ...
loaded.


## Run ablation: 1..N views per scene

For each scene dir, for each N in `FRAME_COUNTS`, uniformly sample N frames and run VGGT. Skips N larger than the scene's available frame count.

In [9]:
rows = []

for scene_dir in SCENE_DIRS:
    scene_path = Path(scene_dir)
    frames = sorted(str(p) for p in scene_path.iterdir() if p.suffix.lower() in IMG_EXTS)
    print(f"\n=== {scene_path.name}: {len(frames)} frames available ===")

    for n in FRAME_COUNTS:
        if n > len(frames):
            print(f"[skip] n={n} > {len(frames)} available frames")
            continue
        sampled = sample_frames(frames, n)
        preds = run_vggt(model, sampled)

        conf = preds["world_points_conf"]
        valid = conf > CONF_THRES
        trans_dist, rot_deg = pairwise_pose_diversity(preds["extrinsic"])

        row = {
            "scene": scene_path.name,
            "n_views": n,
            "depth_conf_mean": float(preds["depth_conf"].mean()),
            "world_conf_mean": float(conf.mean()),
            "frac_valid_points": float(valid.mean()),
            "num_valid_points": int(valid.sum()),
            "mean_cam_translation_dist": trans_dist,
            "mean_cam_rotation_deg": rot_deg,
            "frames": [Path(f).name for f in sampled],
        }

        if SAVE_POINTCLOUD:
            ply_path = OUTPUT_DIR / scene_path.name / f"n{n:02d}.ply"
            save_pointcloud(preds, CONF_THRES, ply_path)
            row["pointcloud"] = str(ply_path)

        rows.append(row)
        print(f"n={n}: world_conf_mean={row['world_conf_mean']:.3f} "
              f"frac_valid={row['frac_valid_points']:.3f} "
              f"cam_trans={trans_dist:.4f} cam_rot_deg={rot_deg:.2f}")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
df


=== image_color: 226 frames available ===
n=1: world_conf_mean=3.582 frac_valid=0.188 cam_trans=nan cam_rot_deg=nan
n=2: world_conf_mean=3.183 frac_valid=0.089 cam_trans=0.5176 cam_rot_deg=92.38
n=3: world_conf_mean=3.242 frac_valid=0.065 cam_trans=0.3492 cam_rot_deg=99.86
n=4: world_conf_mean=3.140 frac_valid=0.048 cam_trans=0.3497 cam_rot_deg=96.38


KeyboardInterrupt: 

## Plot: metric vs. view count (per scene)

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ["world_conf_mean", "frac_valid_points", "mean_cam_translation_dist", "mean_cam_rotation_deg"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for ax, metric in zip(axes.flat, metrics_to_plot):
    for scene, g in df.groupby("scene"):
        ax.plot(g["n_views"], g[metric], marker="o", label=scene)
    ax.set_xlabel("n_views")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)

axes.flat[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "metrics_vs_n_views.png", dpi=150)
plt.show()